In [ ]:
%load_ext autoreload
%autoreload 2
from omegaconf import OmegaConf
from oat.data_module import factory
from oat.model_module.model import OAT
import torch

In [ ]:
from torch.utils.data import DataLoader
from oat.data_module.wind_dataset import QueryBatch
data_config = OmegaConf.load('/scratch/mch/cdonner/observation_assimilation_transformer/configs/data/wind_data.yaml')
dataset = factory.create_dataset(data_config)
#train_dataloader, val_dataloader, test_dataloader = factory.create_dataset(data_config)
#batch = next(iter(train_dataloader))

loader = DataLoader(dataset, batch_size=16, shuffle=False)


In [ ]:
batch = next(iter(loader))

In [ ]:
obs_data = batch['observation_data']
num_stations = obs_data['obs'].shape[1]
key_stations = 100
query = {
    'coords': obs_data['coords'][:, key_stations:, :],
    'topo': obs_data['topo'][:, key_stations:, :],
    'geo': obs_data['geo'][:, key_stations:, :],
    'obs': obs_data['obs'][:, key_stations:, :],
    'mask_obs': obs_data['mask_obs'][:, key_stations:, :],
}
observation_data = {
    'coords': obs_data['coords'][:, :key_stations, :],
    'topo': obs_data['topo'][:, :key_stations, :],
    'geo': obs_data['geo'][:, :key_stations, :],
    'obs': obs_data['obs'][:, :key_stations, :],
    'mask_obs': obs_data['mask_obs'][:, :key_stations, :],
}
batch['observation_data'] = observation_data
batch['query_data'] = query

In [ ]:
model = OAT()

In [ ]:
with torch.no_grad():
    output = model(global_data=batch['global_data'], observation_data=batch['observation_data'], query_data=batch['query_data'], )

In [ ]:

#with torch.no_grad():
# get gradients
pred, loss = model.forward_with_loss(global_data=batch['global_data'], observation_data=batch['observation_data'], query_data=batch['query_data'], target_obs=batch['query_data']['obs'], target_mask_obs=batch['query_data']['mask_obs'])
loss.backward()

In [ ]:
for param in model.parameters():
    if param.grad is not None:
        print(param.grad.abs().sum())

In [ ]:
output

In [ ]:
torch.sum(batch['query_data']['obs'] == torch.nan)

In [ ]:
torch.sum(torch.any(pred['val_pred'] == torch.nan))

In [ ]:
# check if nans in output
torch.sum(output['val_pred'] == torch.nan)

In [ ]:
torch.arange(10)[]

In [ ]:
batch['observation_data']['mask_obs'][2]